# Initialization

In [0]:
import pyspark.sql.functions as f
from  pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import col,trim
from pyspark.sql.window import Window

# Read bronze table

In [0]:
df = spark.table("bikescatalog.bronze.crm_prd_info")

In [0]:
df.display()

# Silver Transformations

# Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))
    

# Product key Parsing

In [0]:
df = df.withColumn("cat_id", f.regexp_replace(f.substring(col("prd_key"),1,5),"-","_"))
df = df.withColumn("prd_key", f.substring(col("prd_key"),7, f.length(col("prd_key"))))

In [0]:
df.display()

# Cost Cleanup

In [0]:
df = df.withColumn("prd_cost", f.coalesce(col("prd_cost"),f.lit(0)))

# Product Line Normalization

In [0]:
df = (
    df
     .withColumn("prd_line",
          f.when(f.upper(col("prd_line")) == "M","Mountain")
            .when(f.upper(col("prd_line")) == 'R',"Road" )
            .when(f.upper(col("prd_line")) == "S", "Other Sales")
            .when(f.upper(col("prd_line")) == "T", "Touring")
            .otherwise("n/a")
    )
)

In [0]:
df.display()

# Date Casting

In [0]:
df = df.withColumn("prd_start_dt", col("prd_start_dt").cast(DateType()))

In [0]:
df.schema.fields

# Renaming Columns

In [0]:
RENAME_MAP = {
    "prd_id": "product_id",
    "cat_id": "category_id",
    "prd_Key": "product_number",
    "prd_nm" : "product_name",
    "prd_cost": "product_cost",
    "prd_line": "product_line",
    "prd_start_dt":"start_date",
    "prd_end_dt": "end_date"
}

for old_name, new_name in RENAME_MAP.items():
  df = df.withColumnRenamed(old_name, new_name)

# Sanity check of dataframe

In [0]:
df.limit(10).display()

# writing silver table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("bikescatalog.silver.crm_products")

# Sanity check of silver table

In [0]:
%sql
select * from bikescatalog.silver.crm_products limit 10;